# SVM + FGM Dual-Stream Consistency Gate Pipeline

This notebook mirrors the LR and NN FGM consistency-gate workflow for an SVM classifier.

Important implementation note:

- FGM needs input gradients.
- A standard sklearn SVM does not provide those gradients.
- This notebook trains a small PyTorch surrogate only to generate FGM adversarial samples.
- The actual clean model, adversarially trained guardian model, evaluations, and consistency gate are all SVM-based.

Pipeline steps:

1. Train a clean SVM on clean training data
2. Train a small PyTorch surrogate and generate FGM adversarial samples
3. Evaluate the clean SVM on clean, adversarial, and combined test data
4. Train a guardian SVM on clean plus FGM samples
5. Evaluate the guardian SVM
6. Run the dual-stream consistency gate


In [19]:
# If needed, install once:
# !pip install torch adversarial-robustness-toolbox scikit-learn pandas numpy joblib

import warnings
warnings.filterwarnings("ignore")

import ast
import random
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim

from art.attacks.evasion import FastGradientMethod
from art.estimators.classification import PyTorchClassifier

try:
    from IPython.display import display
except ImportError:
    display = print


In [20]:
# -----------------------------
# Configuration
# -----------------------------
DATA_PATH = Path(r"..\..\CSVs\dataset.csv")

LABEL_COL = "anomaly"

# Drop non-feature columns if needed
DROP_COLS = {LABEL_COL, "segment", "train", "sampling", "channel"}

TEST_SIZE = 0.2
SEED = 42

# FGM settings
FGM_EPS = 0.10

# For sklearn models, adversarial training is approximated by adding
# FGM samples generated from the surrogate model back into the training set.
ADV_RATIO = 0.55

# Surrogate model settings used only for generating FGM samples.
SURROGATE_HIDDEN = 64
SURROGATE_LR = 1e-3
SURROGATE_BATCH_SIZE = 128
SURROGATE_EPOCHS = 20

# Save paths
SAVE_MODELS = True
ARTIFACT_DIR = Path(r"artifacts")
RESULTS_DIR = Path(r"results")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# SVM settings
BEST_PARAMS_PATH = Path(r"..\..\MachineLearning\SVM\best_params.csv")

DEFAULT_SVM_PARAMS = {
    "C": 1.0,
    "kernel": "rbf",
    "gamma": "scale",
    "degree": 3,
    "class_weight": None,
    "probability": True,
    "random_state": SEED,
}


In [21]:
from sklearn.svm import SVC

def coerce_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        text = value.strip()
        if text.lower() in {"none", "nan", ""}:
            return None
        if text.lower() in {"true", "false"}:
            return text.lower() == "true"
        try:
            return ast.literal_eval(text)
        except Exception:
            return text
    return value


def load_best_params(path: Path, defaults: dict):
    params = defaults.copy()

    if path.exists():
        best_df = pd.read_csv(path)
        row = best_df.iloc[0].to_dict()
        for key, value in row.items():
            if key in params:
                params[key] = coerce_value(value)
        print(f"Loaded SVM params from {path}")
    else:
        print(f"Best params file not found at {path}. Using DEFAULT_SVM_PARAMS.")

    # Required for probability-based consistency gate.
    params["probability"] = True
    params["random_state"] = SEED

    return params


SVM_PARAMS = load_best_params(BEST_PARAMS_PATH, DEFAULT_SVM_PARAMS)
print("SVM_PARAMS:", SVM_PARAMS)


def make_svm_model():
    return SVC(**SVM_PARAMS)


Loaded SVM params from ..\..\MachineLearning\SVM\best_params.csv
SVM_PARAMS: {'C': 1, 'kernel': 'linear', 'gamma': 'scale', 'degree': 3, 'class_weight': None, 'probability': True, 'random_state': 42}


In [22]:
def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in {csv_path}.")

    y = df[LABEL_COL].astype(int).to_numpy()
    feature_df = df[[c for c in df.columns if c not in DROP_COLS]].copy()

    non_numeric_cols = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric_cols:
        raise ValueError(
            f"Non-numeric feature columns found in {csv_path}: {non_numeric_cols}. "
            "Add them to DROP_COLS or encode them before training."
        )

    feature_cols = feature_df.columns.tolist()
    X = feature_df.to_numpy(dtype=np.float32)

    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary labels, got: {np.unique(y)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    print(f"Loaded: {csv_path}")
    print(f"Rows={len(df)}, Features={X.shape[1]}, Label dist={np.bincount(y)}")
    print(f"Train={X_train.shape}, Test={X_test.shape}")

    return X_train, X_test, y_train.astype(np.int64), y_test.astype(np.int64), scaler, feature_cols


if not DATA_PATH.exists():
    raise ValueError(f"Set DATA_PATH first. Current value does not exist: {DATA_PATH}")

X_train, X_test, y_train, y_test, scaler, feature_cols = load_and_prepare(str(DATA_PATH))


Loaded: ..\..\CSVs\dataset.csv
Rows=2123, Features=18, Label dist=[1689  434]
Train=(1698, 18), Test=(425, 18)


In [23]:
class SurrogateMLP(nn.Module):
    def __init__(self, d_in: int, hidden: int = SURROGATE_HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, x):
        return self.net(x)


def make_surrogate_art_classifier(d_in: int):
    model = SurrogateMLP(d_in=d_in)
    loss = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=SURROGATE_LR)

    return PyTorchClassifier(
        model=model,
        loss=loss,
        optimizer=optimizer,
        input_shape=(d_in,),
        nb_classes=2,
        clip_values=(0.0, 1.0),
    )


def predict_labels(model, X: np.ndarray):
    return model.predict(X).astype(int)


def predict_proba(model, X: np.ndarray):
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X)
    else:
        preds = model.predict(X)
        probs = np.zeros((len(preds), 2), dtype=np.float32)
        probs[np.arange(len(preds)), preds.astype(int)] = 1.0
    return probs.astype(np.float32)


def eval_sklearn_classifier(model, X: np.ndarray, y_true: np.ndarray, name: str):
    y_pred = predict_labels(model, X)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }, y_pred


def make_adv_training_set(X_clean, y_clean, X_adv, y_adv, adv_ratio: float = ADV_RATIO):
    n_adv = max(1, int(round(len(X_clean) * adv_ratio)))
    n_adv = min(n_adv, len(X_adv))

    rng = np.random.default_rng(SEED)
    adv_idx = rng.choice(len(X_adv), size=n_adv, replace=False)

    X_mix = np.vstack([X_clean, X_adv[adv_idx]]).astype(np.float32)
    y_mix = np.concatenate([y_clean, y_adv[adv_idx]]).astype(np.int64)

    shuffle_idx = rng.permutation(len(X_mix))
    return X_mix[shuffle_idx], y_mix[shuffle_idx]


In [24]:
# Step 1: train clean SVM model
print("Training clean SVM with:", SVM_PARAMS)

clean_model = make_svm_model()
clean_model.fit(X_train, y_train)

clean_on_clean, y_pred_clean = eval_sklearn_classifier(
    clean_model,
    X_test,
    y_test,
    "clean_model_on_clean_test",
)

if SAVE_MODELS:
    joblib.dump(clean_model, ARTIFACT_DIR / "svm_clean_model.joblib")
    joblib.dump(scaler, ARTIFACT_DIR / "svm_scaler.joblib")


Training clean SVM with: {'C': 1, 'kernel': 'linear', 'gamma': 'scale', 'degree': 3, 'class_weight': None, 'probability': True, 'random_state': 42}

[clean_model_on_clean_test] acc=0.8871 f1=0.6418
confusion matrix:
[[334   4]
 [ 44  43]]
              precision    recall  f1-score   support

           0     0.8836    0.9882    0.9330       338
           1     0.9149    0.4943    0.6418        87

    accuracy                         0.8871       425
   macro avg     0.8992    0.7412    0.7874       425
weighted avg     0.8900    0.8871    0.8734       425



In [25]:
# Step 2: train surrogate and generate FGM adversarial samples
print("Training PyTorch surrogate used only for FGM generation with:", {
    "hidden": SURROGATE_HIDDEN,
    "learning_rate": SURROGATE_LR,
    "batch_size": SURROGATE_BATCH_SIZE,
    "epochs": SURROGATE_EPOCHS,
    "fgm_eps": FGM_EPS,
})

surrogate_art = make_surrogate_art_classifier(d_in=X_train.shape[1])
surrogate_art.fit(
    X_train,
    y_train,
    batch_size=SURROGATE_BATCH_SIZE,
    nb_epochs=SURROGATE_EPOCHS,
)

fgm = FastGradientMethod(estimator=surrogate_art, eps=FGM_EPS)

X_train_adv = fgm.generate(x=X_train).astype(np.float32)
X_test_adv = fgm.generate(x=X_test).astype(np.float32)

print("Adversarial data generated:")
print("X_train_adv:", X_train_adv.shape)
print("X_test_adv:", X_test_adv.shape)

# Keep adversarial labels aligned with original ground-truth labels.
y_train_adv = y_train.copy()
y_test_adv = y_test.copy()

X_test_combined = np.vstack([X_test, X_test_adv]).astype(np.float32)
y_test_combined = np.concatenate([y_test, y_test_adv]).astype(np.int64)


Training PyTorch surrogate used only for FGM generation with: {'hidden': 64, 'learning_rate': 0.001, 'batch_size': 128, 'epochs': 20, 'fgm_eps': 0.1}
Adversarial data generated:
X_train_adv: (1698, 18)
X_test_adv: (425, 18)


In [26]:
# Evaluate clean model on adversarial and combined test data
clean_on_adv, y_pred_adv_clean_model = eval_sklearn_classifier(
    clean_model,
    X_test_adv,
    y_test_adv,
    "clean_model_on_adv_test",
)

clean_on_combined, y_pred_combined_clean_model = eval_sklearn_classifier(
    clean_model,
    X_test_combined,
    y_test_combined,
    "clean_model_on_combined_test",
)



[clean_model_on_adv_test] acc=0.1153 f1=0.1607
confusion matrix:
[[ 13 325]
 [ 51  36]]
              precision    recall  f1-score   support

           0     0.2031    0.0385    0.0647       338
           1     0.0997    0.4138    0.1607        87

    accuracy                         0.1153       425
   macro avg     0.1514    0.2261    0.1127       425
weighted avg     0.1820    0.1153    0.0843       425


[clean_model_on_combined_test] acc=0.5012 f1=0.2715
confusion matrix:
[[347 329]
 [ 95  79]]
              precision    recall  f1-score   support

           0     0.7851    0.5133    0.6208       676
           1     0.1936    0.4540    0.2715       174

    accuracy                         0.5012       850
   macro avg     0.4893    0.4837    0.4461       850
weighted avg     0.6640    0.5012    0.5493       850



In [27]:
# Step 3: adversarial training approximation for SVM
X_train_mixed, y_train_mixed = make_adv_training_set(
    X_train,
    y_train,
    X_train_adv,
    y_train_adv,
    adv_ratio=ADV_RATIO,
)

print("Training guardian SVM with clean + FGM samples:")
print("X_train_mixed:", X_train_mixed.shape)
print("y_train_mixed:", y_train_mixed.shape)

adv_model = make_svm_model()
adv_model.fit(X_train_mixed, y_train_mixed)

if SAVE_MODELS:
    joblib.dump(adv_model, ARTIFACT_DIR / "svm_adversarial_trained_model.joblib")


Training guardian SVM with clean + FGM samples:
X_train_mixed: (2632, 18)
y_train_mixed: (2632,)


In [28]:
# Step 4: evaluate adversarially trained guardian model
adv_trained_on_adv, y_pred_adv = eval_sklearn_classifier(
    adv_model,
    X_test_adv,
    y_test_adv,
    "adv_trained_model_on_adv_test",
)

adv_trained_on_clean, y_pred_clean_adv_model = eval_sklearn_classifier(
    adv_model,
    X_test,
    y_test,
    "adv_trained_model_on_clean_test",
)

adv_trained_on_combined, y_pred_combined_adv_model = eval_sklearn_classifier(
    adv_model,
    X_test_combined,
    y_test_combined,
    "adv_trained_model_on_combined_test",
)



[adv_trained_model_on_adv_test] acc=0.8400 f1=0.3704
confusion matrix:
[[337   1]
 [ 67  20]]
              precision    recall  f1-score   support

           0     0.8342    0.9970    0.9084       338
           1     0.9524    0.2299    0.3704        87

    accuracy                         0.8400       425
   macro avg     0.8933    0.6135    0.6394       425
weighted avg     0.8584    0.8400    0.7982       425


[adv_trained_model_on_clean_test] acc=0.8659 f1=0.5289
confusion matrix:
[[336   2]
 [ 55  32]]
              precision    recall  f1-score   support

           0     0.8593    0.9941    0.9218       338
           1     0.9412    0.3678    0.5289        87

    accuracy                         0.8659       425
   macro avg     0.9003    0.6809    0.7254       425
weighted avg     0.8761    0.8659    0.8414       425


[adv_trained_model_on_combined_test] acc=0.8529 f1=0.4541
confusion matrix:
[[673   3]
 [122  52]]
              precision    recall  f1-score   support


In [29]:
# Summary table
summary_df = pd.DataFrame([
    clean_on_clean,
    clean_on_adv,
    clean_on_combined,
    adv_trained_on_adv,
    adv_trained_on_clean,
    adv_trained_on_combined,
])

display(summary_df)


,model_eval,acc,f1
0,clean_model_on_clean_test,0.887059,0.641791
1,clean_model_on_adv_test,0.115294,0.160714
2,clean_model_on_combined_test,0.501176,0.271478
3,adv_trained_model_on_adv_test,0.840000,0.370370
4,adv_trained_model_on_clean_test,0.865882,0.528926
5,adv_trained_model_on_combined_test,0.852941,0.454148


In [30]:
# Save metrics
summary_path = RESULTS_DIR / "svm_fgm_pipeline_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")


Saved: results\svm_fgm_pipeline_summary.csv


## Dual-Stream Consistency Gate

This section mirrors the dual-stream detector structure from the LR and NN FGM notebooks.

### Gate logic

- **Nominal model** = clean model
- **Guardian model** = adversarially trained model
- Flag likely attacks using:
  1. prediction disagreement
  2. high-confidence disagreement
  3. nominal predicts benign while guardian predicts anomaly by a large enough margin

The gate is evaluated both as:

- a **final prediction system** for clean, adversarial, and combined inputs
- an **attack detector** using **FPR**, **TPR**, and **F1**


In [31]:
CONFIDENCE_THRESHOLD = 0.65
DISAGREEMENT_THRESHOLD = 0.55

def predict_with_confidence(model, X: np.ndarray):
    probs = predict_proba(model, X)
    preds = np.argmax(probs, axis=1)
    return preds, probs


class DualStreamDetector:
    def __init__(self, nominal_model, guardian_model):
        self.nominal_model = nominal_model
        self.guardian_model = guardian_model

    def detect_attacks(
        self,
        X: np.ndarray,
        confidence_threshold: float = CONFIDENCE_THRESHOLD,
        disagreement_threshold: float = DISAGREEMENT_THRESHOLD,
    ):
        preds_nominal, probs_nominal = predict_with_confidence(self.nominal_model, X)
        preds_guardian, probs_guardian = predict_with_confidence(self.guardian_model, X)

        n_samples = len(X)
        flags = np.zeros(n_samples, dtype=int)
        details = []

        for i in range(n_samples):
            yA = int(preds_nominal[i])
            yB = int(preds_guardian[i])
            pA = probs_nominal[i]
            pB = probs_guardian[i]

            conf_nominal = float(pA[yA])
            conf_guardian = float(pB[yB])

            detected = False
            reasons = []

            if yA != yB:
                detected = True
                reasons.append("disagreement")

            if yA == 0 and yB == 1:
                prob_diff = float(pB[1] - pA[1])
                if conf_nominal >= confidence_threshold and conf_guardian >= confidence_threshold:
                    if prob_diff >= disagreement_threshold:
                        detected = True
                        reasons.append("high_confidence_disagreement")
            else:
                prob_diff = float(pB[1] - pA[1])

            flags[i] = int(detected)
            details.append({
                "nominal_pred": yA,
                "guardian_pred": yB,
                "nominal_conf": conf_nominal,
                "guardian_conf": conf_guardian,
                "nominal_anom_prob": float(pA[1]),
                "guardian_anom_prob": float(pB[1]),
                "prob_diff_anomaly": prob_diff,
                "detected": bool(detected),
                "reason": ",".join(reasons) if reasons else "none",
            })

        final_preds = np.where(flags == 1, preds_guardian, preds_nominal)
        details_df = pd.DataFrame(details)
        return final_preds, flags, details_df


def evaluate_dual_stream_predictions(y_true: np.ndarray, y_pred: np.ndarray, name: str):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }


detector = DualStreamDetector(
    nominal_model=clean_model,
    guardian_model=adv_model,
)

gated_clean_pred, flags_clean, gate_clean_df = detector.detect_attacks(X_test)
gated_adv_pred, flags_adv, gate_adv_df = detector.detect_attacks(X_test_adv)
gated_combined_pred, flags_combined, gate_combined_df = detector.detect_attacks(X_test_combined)

gated_clean_metrics = evaluate_dual_stream_predictions(
    y_test, gated_clean_pred, "dual_stream_final_predictions_on_clean_test"
)
gated_adv_metrics = evaluate_dual_stream_predictions(
    y_test_adv, gated_adv_pred, "dual_stream_final_predictions_on_adv_test"
)
gated_combined_metrics = evaluate_dual_stream_predictions(
    y_test_combined, gated_combined_pred, "dual_stream_final_predictions_on_combined_test"
)

dual_stream_prediction_summary_df = pd.DataFrame([
    gated_clean_metrics,
    gated_adv_metrics,
    gated_combined_metrics,
])

print("\nDual-stream final prediction summary:")
display(dual_stream_prediction_summary_df)

y_attack_true = np.concatenate([
    np.zeros(len(flags_clean), dtype=int),
    np.ones(len(flags_adv), dtype=int),
])
y_attack_pred = np.concatenate([flags_clean, flags_adv])

dual_stream_detection_results = pd.DataFrame([{
    "attack": "FGM",
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "disagreement_threshold": DISAGREEMENT_THRESHOLD,
    "FPR": float(flags_clean.mean()),
    "TPR": float(flags_adv.mean()),
    "F1": float(f1_score(y_attack_true, y_attack_pred, zero_division=0)),
    "clean_model_acc_on_adv": float(clean_on_adv["acc"]),
    "guardian_model_acc_on_clean": float(adv_trained_on_clean["acc"]),
    "guardian_model_acc_on_adv": float(adv_trained_on_adv["acc"]),
}])

print("\nDual-stream attack-detection summary:")
display(dual_stream_detection_results)

print("\nGate reason counts on clean test:")
display(gate_clean_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))

print("\nGate reason counts on adversarial test:")
display(gate_adv_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))



[dual_stream_final_predictions_on_clean_test] acc=0.8729 f1=0.5645
confusion matrix:
[[336   2]
 [ 52  35]]
              precision    recall  f1-score   support

           0     0.8660    0.9941    0.9256       338
           1     0.9459    0.4023    0.5645        87

    accuracy                         0.8729       425
   macro avg     0.9060    0.6982    0.7451       425
weighted avg     0.8823    0.8729    0.8517       425


[dual_stream_final_predictions_on_adv_test] acc=0.8565 f1=0.4874
confusion matrix:
[[335   3]
 [ 58  29]]
              precision    recall  f1-score   support

           0     0.8524    0.9911    0.9166       338
           1     0.9062    0.3333    0.4874        87

    accuracy                         0.8565       425
   macro avg     0.8793    0.6622    0.7020       425
weighted avg     0.8634    0.8565    0.8287       425


[dual_stream_final_predictions_on_combined_test] acc=0.8647 f1=0.5267
confusion matrix:
[[671   5]
 [110  64]]
              prec

,model_eval,acc,f1
0,dual_stream_final_predictions_on_clean_test,0.872941,0.564516
1,dual_stream_final_predictions_on_adv_test,0.856471,0.487395
2,dual_stream_final_predictions_on_combined_test,0.864706,0.526749



Dual-stream attack-detection summary:


,attack,confidence_threshold,disagreement_threshold,FPR,TPR,F1,clean_model_acc_on_adv,guardian_model_acc_on_clean,guardian_model_acc_on_adv
0,FGM,0.65,0.55,0.077647,0.863529,0.889697,0.115294,0.865882,0.84



Gate reason counts on clean test:


,reason,count
0,none,392
1,disagreement,33



Gate reason counts on adversarial test:


,reason,count
0,disagreement,365
1,none,58
2,"disagreement,high_confidence_disagreement",2


In [32]:
# Save dual-stream outputs
dual_stream_prediction_summary_path = RESULTS_DIR / "svm_dual_stream_prediction_summary.csv"
dual_stream_detection_path = RESULTS_DIR / "svm_dual_stream_detection_results.csv"

dual_stream_prediction_summary_df.to_csv(dual_stream_prediction_summary_path, index=False)
dual_stream_detection_results.to_csv(dual_stream_detection_path, index=False)

gate_clean_df.to_csv(RESULTS_DIR / "svm_dual_stream_gate_clean_details.csv", index=False)
gate_adv_df.to_csv(RESULTS_DIR / "svm_dual_stream_gate_adv_details.csv", index=False)
gate_combined_df.to_csv(RESULTS_DIR / "svm_dual_stream_gate_combined_details.csv", index=False)

print(f"Saved: {dual_stream_prediction_summary_path}")
print(f"Saved: {dual_stream_detection_path}")


Saved: results\svm_dual_stream_prediction_summary.csv
Saved: results\svm_dual_stream_detection_results.csv
